# Lesson 10A: Topic Modeling Theory — Latent Dirichlet Allocation and Gibbs Sampling

## Introduction

Every method so far assigns each data point to (at most) one cluster, or projects it onto a small number of continuous axes. Documents don't fit that mold well: a news article about "the economics of professional sports" is genuinely *about* two things at once, in different proportions. **Latent Dirichlet Allocation (LDA)** (Blei, Ng, Jordan, 2003) models exactly that: each document is a **mixture of topics**, and each topic is a **distribution over words** — discovered entirely from word co-occurrence, with no topic labels ever provided.

This is the discrete-data, generative-model counterpart to GMM (Lesson 4): where GMM models each point as generated by one of several Gaussians with soft (probabilistic) cluster assignment, LDA models each *word* in a document as generated by one of several topics, with soft topic proportions per document.

The catch: computing the exact posterior over topic assignments is **intractable** — it requires summing over every possible topic assignment for every word in every document, which is exponential. LDA is a textbook case for **approximate inference**, and this lesson derives one of the two classic approaches: **collapsed Gibbs sampling**.

In this lesson:
1. Frame documents as bags-of-words and build the document-term matrix
2. Derive LDA's generative story: Dirichlet priors over document-topic and topic-word distributions
3. Understand why exact inference is intractable
4. Derive collapsed Gibbs sampling: integrate out the continuous parameters, sample only discrete topic assignments
5. Implement collapsed Gibbs sampling from scratch in NumPy
6. Cross-check the recovered topics against scikit-learn's variational-Bayes LDA

## Table of Contents

1. [Introduction](#introduction)
2. [Required Libraries](#required-libraries)
3. [Bag-of-Words and the Document-Term Matrix](#bow)
4. [LDA's Generative Model](#generative-model)
5. [Why Exact Inference Is Intractable](#intractable)
6. [Collapsed Gibbs Sampling](#gibbs-derivation)
7. [From-Scratch Implementation](#gibbs-scratch)
8. [Cross-Check Against scikit-learn](#sklearn-check)
9. [Visualizing Topics](#visualization)
10. [Conclusion](#conclusion)

<a name="required-libraries"></a>
## Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✅ Libraries loaded!")

<a name="bow"></a>
## Bag-of-Words and the Document-Term Matrix

LDA (like most classic topic models) treats each document as a **bag of words** — word order is discarded, only word *counts* matter. A corpus of $D$ documents over a vocabulary of $V$ words becomes a $D \times V$ **document-term matrix**, where entry $(d, v)$ is how many times word $v$ appears in document $d$.

### A Synthetic Corpus with Known Topics

We generate documents *from LDA's own generative process*, using three clearly-separated vocabularies (sports, cooking, finance). This gives us ground truth: a correct implementation should recover these three topics from word co-occurrence alone, with no topic labels ever provided to either model.

In [ ]:
rng = np.random.RandomState(42)

topic_vocab = {
    'sports': ['ball', 'game', 'team', 'score', 'player', 'coach', 'match', 'win'],
    'cooking': ['recipe', 'oven', 'bake', 'flavor', 'ingredient', 'chef', 'dish', 'spice'],
    'finance': ['stock', 'market', 'invest', 'profit', 'bank', 'trade', 'shares', 'fund'],
}
true_topic_names = list(topic_vocab.keys())
K = len(true_topic_names)
vocab = sorted({w for words in topic_vocab.values() for w in words})
V = len(vocab)
word_to_idx = {w: i for i, w in enumerate(vocab)}

alpha_true, beta_true = 0.3, 0.1

# True topic-word distributions: near-uniform over the topic's own vocabulary, smoothed
phi_true = np.zeros((K, V))
for k, topic in enumerate(true_topic_names):
    for w in topic_vocab[topic]:
        phi_true[k, word_to_idx[w]] = 1.0
    phi_true[k] += beta_true
    phi_true[k] /= phi_true[k].sum()

n_docs, doc_length = 300, 40
documents = []
for d in range(n_docs):
    theta_d = rng.dirichlet(np.full(K, alpha_true))
    words = []
    for _ in range(doc_length):
        z = rng.choice(K, p=theta_d)
        w = rng.choice(V, p=phi_true[z])
        words.append(vocab[w])
    documents.append(' '.join(words))

vectorizer = CountVectorizer(vocabulary=vocab)
dtm = vectorizer.fit_transform(documents).toarray()

print(f"Corpus: {n_docs} documents, vocabulary size {V}, document-term matrix shape {dtm.shape}")
print(f"Example document: {documents[0][:120]}...")

<a name="generative-model"></a>
## LDA's Generative Model

LDA assumes every document was generated by this process:

1. For each topic $k = 1, \ldots, K$: draw a word distribution $\phi_k \sim \text{Dirichlet}(\beta)$ over the vocabulary
2. For each document $d = 1, \ldots, D$: draw a topic distribution $\theta_d \sim \text{Dirichlet}(\alpha)$ over the $K$ topics
3. For each word position $i$ in document $d$:
   - draw a topic assignment $z_{d,i} \sim \text{Categorical}(\theta_d)$
   - draw the word $w_{d,i} \sim \text{Categorical}(\phi_{z_{d,i}})$

### Why Dirichlet Priors

The Dirichlet distribution is the natural prior for categorical/multinomial parameters — it's **conjugate** to the categorical likelihood, meaning the posterior stays in the same family. This conjugacy is exactly what makes collapsed Gibbs sampling tractable, as we'll see.

- $\alpha$ controls document-topic sparsity: small $\alpha$ pushes each document toward a few dominant topics; large $\alpha$ spreads a document more evenly across all $K$ topics.
- $\beta$ controls topic-word sparsity: small $\beta$ pushes each topic toward a small set of characteristic words; large $\beta$ spreads a topic more evenly across the vocabulary.

### The Inference Problem

Given only the observed words $w$, we want the posterior over the latent variables: $P(\theta, \phi, z \mid w, \alpha, \beta)$. $\theta$ and $\phi$ tell us document-topic and topic-word proportions; $z$ tells us which topic generated each individual word.

<a name="intractable"></a>
## Why Exact Inference Is Intractable

The posterior requires normalizing over every possible configuration of topic assignments across every word in every document — $K^N$ configurations for $N$ total word tokens. For even a modest corpus (hundreds of documents, tens of words each), this is astronomically large. There is no closed form.

Two families of approximate inference make LDA practical:
- **Variational inference** (the original Blei et al. approach, and what scikit-learn's `LatentDirichletAllocation` uses by default): optimize a tractable approximating distribution to be as close as possible to the true posterior.
- **Gibbs sampling** (Griffiths & Steyvers, 2004): construct a Markov chain whose stationary distribution *is* the true posterior, and sample from it directly.

This lesson derives the second — specifically **collapsed** Gibbs sampling, which analytically integrates out $\theta$ and $\phi$ and samples only the discrete topic assignments $z$, converging faster than sampling all three families of latent variables jointly.

<a name="gibbs-derivation"></a>
## Collapsed Gibbs Sampling

### Collapsing Out $\theta$ and $\phi$

Because the Dirichlet is conjugate to the categorical/multinomial, $\theta$ and $\phi$ can be integrated out of the joint distribution analytically, leaving a distribution over $z$ alone: $P(z \mid w, \alpha, \beta)$. Sampling directly from this collapsed posterior mixes faster than sampling $\theta$, $\phi$, and $z$ jointly.

### The Full Conditional

Gibbs sampling needs the conditional distribution of each variable given all others. For the topic assignment of the $i$-th word in document $d$, holding every other assignment fixed:

$$P(z_{d,i} = k \mid z_{-(d,i)}, w) \propto \frac{n_{d,k}^{-(d,i)} + \alpha}{\;\cdot\;} \cdot \frac{n_{k,w_{d,i}}^{-(d,i)} + \beta}{n_k^{-(d,i)} + V\beta}$$

where (all counts exclude the current word being resampled, denoted by the $-(d,i)$ superscript):
- $n_{d,k}$ = number of words in document $d$ currently assigned to topic $k$
- $n_{k,v}$ = number of times word $v$ is assigned to topic $k$ across the whole corpus
- $n_k$ = total words currently assigned to topic $k$

**Reading the two factors**: the first term says "how much does document $d$ already favor topic $k$?" (document-topic affinity); the second says "how well does topic $k$ explain this specific word?" (topic-word affinity). A word gets reassigned to whichever topic is favored by both its document's current topic mix and its own word identity.

### The Sampling Loop

1. Randomly initialize every word's topic assignment
2. Repeat for many iterations: for every word, remove its current assignment from the counts, compute the conditional distribution above, sample a new assignment, add it back to the counts
3. After the chain mixes (burn-in), estimate:

$$\hat\theta_{d,k} = \frac{n_{d,k} + \alpha}{n_d + K\alpha}, \qquad \hat\phi_{k,v} = \frac{n_{k,v} + \beta}{n_k + V\beta}$$

<a name="gibbs-scratch"></a>
## From-Scratch Implementation

In [ ]:
def initialize_gibbs_state(dtm: np.ndarray, K: int, rng: np.random.RandomState):
    n_docs, V = dtm.shape
    doc_tokens = [np.repeat(np.arange(V), dtm[d]) for d in range(n_docs)]
    z = [rng.randint(0, K, size=len(doc_tokens[d])) for d in range(n_docs)]

    doc_topic_counts = np.zeros((n_docs, K), dtype=int)
    topic_word_counts = np.zeros((K, V), dtype=int)
    topic_counts = np.zeros(K, dtype=int)

    for d in range(n_docs):
        for i, w in enumerate(doc_tokens[d]):
            k = z[d][i]
            doc_topic_counts[d, k] += 1
            topic_word_counts[k, w] += 1
            topic_counts[k] += 1

    return doc_tokens, z, doc_topic_counts, topic_word_counts, topic_counts


def collapsed_gibbs_lda(dtm: np.ndarray, K: int, alpha: float, beta: float, n_iter: int, rng: np.random.RandomState):
    n_docs, V = dtm.shape
    doc_tokens, z, doc_topic_counts, topic_word_counts, topic_counts = initialize_gibbs_state(dtm, K, rng)

    for _ in range(n_iter):
        for d in range(n_docs):
            tokens, zd = doc_tokens[d], z[d]
            for i in range(len(tokens)):
                w = tokens[i]
                k_old = zd[i]

                # Remove this word's current assignment before resampling
                doc_topic_counts[d, k_old] -= 1
                topic_word_counts[k_old, w] -= 1
                topic_counts[k_old] -= 1

                conditional = (doc_topic_counts[d] + alpha) * (topic_word_counts[:, w] + beta) / (topic_counts + V * beta)
                conditional /= conditional.sum()
                k_new = rng.choice(K, p=conditional)

                zd[i] = k_new
                doc_topic_counts[d, k_new] += 1
                topic_word_counts[k_new, w] += 1
                topic_counts[k_new] += 1

    theta = (doc_topic_counts + alpha) / (doc_topic_counts.sum(axis=1, keepdims=True) + K * alpha)
    phi = (topic_word_counts + beta) / (topic_counts[:, None] + V * beta)
    return theta, phi


alpha, beta = 0.3, 0.1
theta_scratch, phi_scratch = collapsed_gibbs_lda(dtm, K=K, alpha=alpha, beta=beta, n_iter=200, rng=rng)

print("From-scratch collapsed Gibbs sampling — top words per recovered topic:")
for k in range(K):
    top_idx = np.argsort(-phi_scratch[k])[:8]
    print(f"  Topic {k}: {[vocab[i] for i in top_idx]}")

<a name="sklearn-check"></a>
## Cross-Check Against scikit-learn

Scikit-learn's `LatentDirichletAllocation` fits the same generative model with **online variational Bayes** — a fundamentally different approximate-inference algorithm than collapsed Gibbs sampling. The two won't produce numerically identical $\theta$/$\phi$ values (different algorithms, different local optima, and topic *indices* are arbitrary — topic 0 in one model might be topic 2 in the other). The meaningful cross-check is whether **both recover the same underlying topics** — the same word groupings, however labeled.

In [ ]:
lda_sklearn = LatentDirichletAllocation(n_components=K, random_state=42, max_iter=50, learning_method='batch')
lda_sklearn.fit(dtm)
phi_sklearn = lda_sklearn.components_ / lda_sklearn.components_.sum(axis=1, keepdims=True)

print("scikit-learn (variational Bayes) — top words per recovered topic:")
for k in range(K):
    top_idx = np.argsort(-phi_sklearn[k])[:8]
    print(f"  Topic {k}: {[vocab[i] for i in top_idx]}")

print(f"\nTrue generating vocabularies: {topic_vocab}")


def top_words_set(phi_row: np.ndarray, n: int = 8) -> set:
    return set(vocab[i] for i in np.argsort(-phi_row)[:n])


# Match topics across models by maximum top-word overlap, since topic indices are arbitrary
scratch_sets = [top_words_set(phi_scratch[k]) for k in range(K)]
sklearn_sets = [top_words_set(phi_sklearn[k]) for k in range(K)]

overlaps = np.zeros((K, K), dtype=int)
for i in range(K):
    for j in range(K):
        overlaps[i, j] = len(scratch_sets[i] & sklearn_sets[j])

best_match_overlap = overlaps.max(axis=1)
print(f"\nBest top-8-word overlap per from-scratch topic against its best-matching sklearn topic: {best_match_overlap.tolist()}")
print("💡 Both algorithms — collapsed Gibbs sampling and variational Bayes — independently recover the same three topics from word co-occurrence alone")

<a name="visualization"></a>
## Visualizing Topics

Plot the top words per topic (from our from-scratch model) as horizontal bar charts, and the document-topic proportions $\theta$ as a heatmap over a sample of documents — showing that most documents concentrate on one or two topics, as the generative Dirichlet prior with small $\alpha$ would predict.

In [ ]:
fig, axes = plt.subplots(1, K, figsize=(5 * K, 4))
for k in range(K):
    top_idx = np.argsort(phi_scratch[k])[-8:]
    axes[k].barh([vocab[i] for i in top_idx], phi_scratch[k][top_idx], color='steelblue')
    axes[k].set_title(f'Topic {k} — top words', fontsize=11, fontweight='bold')
    axes[k].set_xlabel('P(word | topic)')
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 6))
sample_docs = theta_scratch[:30]
im = ax.imshow(sample_docs, aspect='auto', cmap='YlOrRd')
ax.set_xlabel('Topic')
ax.set_ylabel('Document (first 30)')
ax.set_xticks(range(K))
ax.set_title('Document-Topic Proportions ($\\hat\\theta$)', fontsize=12, fontweight='bold')
plt.colorbar(im, ax=ax, label='P(topic | document)')
plt.tight_layout()
plt.show()

print("💡 Most documents concentrate their probability mass on one topic — a direct consequence of the small alpha=0.3 used to generate this corpus")

<a name="conclusion"></a>
## Conclusion

### Key Takeaways

1. **LDA models documents as topic mixtures, not single-cluster members** — the continuous, probabilistic sibling of hard clustering, applied to discrete word data.
2. **Dirichlet priors over document-topic and topic-word distributions** give LDA its generative story, and their conjugacy to the categorical likelihood is what makes efficient inference possible at all.
3. **Exact inference is intractable** — $K^N$ possible topic-assignment configurations rules out any closed-form posterior.
4. **Collapsed Gibbs sampling integrates out $\theta$ and $\phi$ analytically**, leaving a simple, interpretable conditional distribution over topic assignments that balances document-topic affinity against topic-word affinity.
5. **Different inference algorithms (Gibbs vs. variational) recover the same latent topics** even though their numeric outputs and topic indices don't align — a healthy sign that the model, not the inference method, is what determines what gets discovered.

### When to Use LDA

- ✅ Discovering latent themes in a large, unlabeled document collection
- ✅ Documents that genuinely mix multiple themes (news articles, research abstracts, product reviews)
- ❌ Very short documents (tweets, single sentences) — too few word co-occurrences per document for reliable topic estimates
- ❌ When topic labels already exist — supervised classification will do better with less machinery

### Next Steps

In Lesson 10B (practical), we'll:
- Apply Gensim's and scikit-learn's production LDA implementations to a larger, real-world-style corpus
- Use `pyLDAvis` for interactive topic visualization
- Discuss choosing the number of topics $K$ and interpreting topic coherence

You now understand how a generative probabilistic model recovers latent structure from purely discrete, unlabeled data — the same spirit as GMM's soft clustering, applied to an entirely different data type 🎯